# 12 — Multi-head Latent Attention (MLA)

**Before this:** notebook **11** (roadmap).

**After this:** notebook **13** (MoE).

**Paper:** DeepSeek-V2 — [arxiv](https://arxiv.org/html/2405.04434)

---

## What you already know (GPT-2)

Notebook **4** + `llmc.model.CausalSelfAttention`:

- `c_attn` maps `x` → **Q, K, V** in one shot (width `3C`).
- Attention: `softmax(Q K^T / sqrt(d)) V` with a causal mask.
- At inference, KV cache stores **full K and V** per head.

## What MLA changes (one idea)

Store a **small latent** `c_kv` per token instead of full K,V. Reconstruct K,V when you need them.

**C port:** `c/deepseek_v2/mla.c` — run `make test_mla`.


## Picture (one token, one layer)

```text
x  ──► wq ──► Q  ──┐
                   ├──► attention ──► wo ──► output
x  ──► w_dkv ──► c_kv ──► w_uk ──► K ──┤
              └──► w_uv ──► V ──┘

Cache at inference: mainly c_kv  (dim = kv_lora_rank)
GPT-2 cache:         K and V     (dim = 2 * n_head * head_dim)
```


In [ ]:
# --- Cell: run MLA module (read llmc/deepseek_v2.py in parallel) ---
import torch
from llmc.deepseek_v2 import DeepSeekV2Config, MultiHeadLatentAttention

cfg = DeepSeekV2Config.tiny(vocab_size=128, block_size=32)
x = torch.randn(2, 16, cfg.n_embd)

attn = MultiHeadLatentAttention(cfg)
y = attn(x)

print("Step-by-step shapes:")
print("  x:     ", tuple(x.shape), "   # (B, T, C)")
print("  c_kv:  ", tuple(attn.last_c_kv.shape), "   # (B, T, kv_lora_rank)  <-- cache this at inference")
print("  y:     ", tuple(y.shape), "   # (B, T, C)")
print("  head_dim:", cfg.n_embd // cfg.n_head, "| kv_lora_rank:", cfg.kv_lora_rank)


In [ ]:
# --- Compare to GPT-2 attention on the SAME x (educational diff) ---
from llmc.model import GPTConfig, CausalSelfAttention

gcfg = GPTConfig.tiny(128, 32)
gcfg.n_embd = cfg.n_embd
gcfg.n_head = cfg.n_head
gattn = CausalSelfAttention(gcfg)

# GPT-2: one Linear  C -> 3C
print("GPT-2 c_attn weight shape:", tuple(gattn.c_attn.weight.shape), "  # (3C, C)")
print("MLA wq shape:", tuple(attn.wq.weight.shape))
print("MLA w_dkv shape:", tuple(attn.w_dkv.weight.shape), "  # (kv_lora_rank, C)")


In [ ]:
# --- KV cache size (why MLA matters for long context) ---
from llmc.deepseek_v2 import DeepSeekV2

model = DeepSeekV2(cfg)
mla_b = model.kv_cache_bytes_per_token()
mha_b = model.mha_kv_cache_bytes_per_token()
print("Per token (all layers, float32):")
print("  MLA cache:", mla_b, "bytes")
print("  MHA cache:", mha_b, "bytes")
print("  MHA is %.1fx larger" % (mha_b / mla_b))


## C exercise

```bash
cd c && make test_mla && ./bin/test_mla
```

Open `mla.c` next to `MultiHeadLatentAttention.forward` — same four passes as `attention_forward` in llm.c, but K/V come from `c_kv`.

**Next:** notebook **13** (MoE). GPT-2 MLP in llm.c has **no router**.
